# 00 · Colab setup + Part A checks (A1, A2)

Run top to bottom on a fresh runtime (T4 is enough for this notebook).
1. Mount Drive, clone the repo, install the pinned environment.
2. Download LIBERO assets and the T1–T4 demos to Drive (skipped if already there).
3. **A1:** render RGB, depth and segmentation from both cameras, check they are upright.
4. **A2:** replay expert demos through our wrapper; every episode must succeed.

Persistent files go to `MyDrive/groundedvla/` (plan Section 2.6). Save this notebook after running so outputs show up in VS Code.

In [1]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

commit 8f27027  Part A: evaluation metrics, dataset build, 3D grounding, camera perturbation (A4-A7)


## Install (≈3–5 min)

In [5]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
!bash scripts/colab_install.sh requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for egl_probe (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error origina

In [ ]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

## LIBERO assets + demos for T1–T4 → Drive

In [ ]:
!python scripts/download_all.py --what assets data

## A1 · Render check (RGB, metric depth, instance segmentation)

In [ ]:
!python scripts/smoke_test_env.py --task T1 --steps 20
!python scripts/smoke_test_env.py --task T3 --steps 20

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
smoke = Path(paths["outputs"]) / "smoke"
names = ["agentview", "wrist", "agentview_depth", "agentview_seg"]
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for r, task in enumerate(["T1", "T3"]):
    for c, n in enumerate(names):
        ax[r, c].imshow(Image.open(smoke / f"{task}_{n}.png")); ax[r, c].set_title(f"{task} {n}"); ax[r, c].axis("off")
plt.show()   # all images must be upright: table at the bottom, robot at the back

## A2 · Expert replay through the wrapper (must be 100%)

In [ ]:
for task in ["T1", "T2", "T3"]:
    !python scripts/run_sim.py replay --task {task} --episodes 5 --viewer none --save-video

In [ ]:
from pathlib import Path
from IPython.display import Video, display
for v in sorted(Path(paths["outputs"]).glob("run_sim/replay/videos/*.mp4"))[:3]:
    print(v.name); display(Video(str(v), embed=True, width=512))

In [ ]:
import json
import pandas as pd
rows = []
for f in Path(paths["outputs"], "run_sim").glob("*/*.jsonl"):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
df.groupby(["variant", "task"]).agg(episodes=("success", "size"), success=("success", "mean"),
                                    steps=("steps", "mean"), policy_call_ms=("latency_ms_median", "median"))